# SLAVA v0: collection and review of 102 simulation scenes

Этот notebook собирает environment-first inventory:

- 30 LIBERO tasks × init states `[0, 17, 34]` = 90 сцен;
- 4 SimplerEnv bridge tasks × episode IDs `[0, 8, 16]` = 12 сцен;
- всего 102 строки `task × init_state`.

Симуляторы запускаются в отдельных virtual environments. Notebook хранит объединённый DataFrame, интерактивную разметку и экспорт JSONL/CSV. Сбор работает в resume-режиме.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import pandas as pd
from IPython.display import display

def find_project_root():
    candidates = [os.environ.get('SLAVA_ROOT'), Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if not candidate:
            continue
        candidate = Path(candidate).expanduser().resolve()
        if (candidate / 'src' / 'slava_inventory').is_dir():
            return candidate
    raise RuntimeError('Не найден корень SLAVA_dev. Откройте папку проекта в VS Code перед запуском ноутбука.')

PROJECT_ROOT = find_project_root()
DEPS_DIR = Path(os.environ.get('SLAVA_DEPS_DIR', PROJECT_ROOT.parent)).expanduser().resolve()
LIBERO_REPO = Path(os.environ.get('LIBERO_ROOT', DEPS_DIR / 'LIBERO')).expanduser().resolve()
SIMPLER_REPO = Path(os.environ.get('SIMPLERENV_ROOT', DEPS_DIR / 'SimplerEnv')).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from slava_inventory.notebook_ui import (
    InventoryReviewer,
    LexiconReviewer,
    create_or_update_lexicon,
    export_inventory_dataframe,
    merge_inventories,
)

conda_candidates = [
    os.environ.get('CONDA_EXE'),
    shutil.which('conda'),
    '/opt/miniforge3/bin/conda',
    '/opt/conda/bin/conda',
]
CONDA_EXE = next((str(Path(p)) for p in conda_candidates if p and Path(p).is_file()), None)
LIBERO_CONDA_ENV = 'slava-libero'
SIMPLER_CONDA_ENV = 'slava-simpler'
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Conda:', CONDA_EXE or 'NOT FOUND')

## 1. Проверка путей и окружений

Если Conda environment отсутствует, сначала выполните инструкции из `README.md`.

In [ ]:
assert PROJECT_ROOT.exists(), PROJECT_ROOT
assert LIBERO_REPO.exists(), LIBERO_REPO
assert SIMPLER_REPO.exists(), SIMPLER_REPO

if CONDA_EXE is None:
    print('WARNING: Conda executable was not found. See README.md.')
    conda_envs = {}
else:
    result = subprocess.run(
        [CONDA_EXE, 'env', 'list', '--json'], capture_output=True, text=True, check=True
    )
    env_paths = json.loads(result.stdout)['envs']
    conda_envs = {Path(path).name: path for path in env_paths}
    display(pd.DataFrame(sorted(conda_envs.items()), columns=['environment', 'path']))

for env_name in [LIBERO_CONDA_ENV, SIMPLER_CONDA_ENV]:
    if env_name not in conda_envs:
        print(f'WARNING: Conda environment {env_name!r} was not found')

## 2. Сбор сцен

По умолчанию collectors не запускаются. Поменяйте нужный флаг на `True`.

- Повторный запуск безопасен: существующие `task_uid` пропускаются.
- `OVERWRITE_EXISTING=True` полностью пересоздаст соответствующий partial inventory.
- Ошибки отдельных сцен сохраняются в `data/collection_errors.jsonl`.
- Для LIBERO headless renderer запускается с `MUJOCO_GL=egl`.

In [ ]:
RUN_LIBERO = False
RUN_SIMPLER = False
OVERWRITE_EXISTING = False
FAIL_FAST = False

def run_streaming(command, extra_env=None):
    env = None
    if extra_env:
        import os
        env = os.environ.copy()
        env.update(extra_env)
    print(' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=PROJECT_ROOT, env=env, check=True)

if (RUN_LIBERO or RUN_SIMPLER) and CONDA_EXE is None:
    raise RuntimeError('Conda executable was not found. See README.md.')

if RUN_LIBERO:
    cmd = [
        CONDA_EXE, 'run', '--no-capture-output', '-n', LIBERO_CONDA_ENV,
        'python', PROJECT_ROOT / 'scripts/collect_libero.py',
        '--libero-repo', LIBERO_REPO,
        '--output-root', DATA_DIR,
        '--init-state-ids', 0, 17, 34,
        '--image-size', 256,
        '--settle-steps', 0,
    ]
    if OVERWRITE_EXISTING:
        cmd.append('--overwrite')
    if FAIL_FAST:
        cmd.append('--fail-fast')
    run_streaming(cmd, {'MUJOCO_GL': 'egl', 'MUJOCO_EGL_DEVICE_ID': '0'})

if RUN_SIMPLER:
    cmd = [
        CONDA_EXE, 'run', '--no-capture-output', '-n', SIMPLER_CONDA_ENV,
        'python', PROJECT_ROOT / 'scripts/collect_simpler.py',
        '--simpler-repo', SIMPLER_REPO,
        '--output-root', DATA_DIR,
        '--episode-ids', 0, 8, 16,
    ]
    if OVERWRITE_EXISTING:
        cmd.append('--overwrite')
    if FAIL_FAST:
        cmd.append('--fail-fast')
    run_streaming(cmd)

## 3. Объединение partial inventories

При повторном объединении ручные annotations из существующего `task_inventory.jsonl` сохраняются.

In [ ]:
inventory_df = merge_inventories(DATA_DIR)
print('Total scenes:', len(inventory_df))
if inventory_df.empty:
    print('Inventory is empty. Set RUN_LIBERO/RUN_SIMPLER=True and run the collection cell.')
else:
    display(inventory_df.groupby('suite').size().rename('scenes').to_frame())
    display(inventory_df[['task_uid', 'suite', 'canonical_en', 'review_status', 'usable_for_slava']].head())
    if len(inventory_df) != 102:
        print('WARNING: expected 102 rows. Inspect data/collection_errors.jsonl and rerun collectors.')

## 4. Интерактивный review сцен

Для каждой сцены можно:

- отметить `usable_for_slava`;
- поставить галочку `selected_for_v0`;
- записать причины исключения и заметки;
- заполнить candidate semantic slots;
- вручную отметить видимость каждого sim object в agent/wrist view.

Кнопки Previous/Next сначала сохраняют текущую форму в DataFrame. Кнопка Export JSONL атомарно записывает DataFrame в `data/task_inventory.jsonl`.

In [ ]:
scene_reviewer = None
if inventory_df.empty:
    print('Scene review is unavailable until the inventory has been collected.')
else:
    scene_reviewer = InventoryReviewer(inventory_df, DATA_DIR)
    scene_reviewer.show()

После работы с формой используйте DataFrame из reviewer:

In [ ]:
if scene_reviewer is None:
    print('No scene-review summary yet.')
else:
    inventory_df = scene_reviewer.df
    summary = pd.DataFrame({
        'value': [
            len(inventory_df),
            int((inventory_df.review_status != 'pending').sum()),
            int((inventory_df.usable_for_slava == True).sum()),
            int(inventory_df.selected_for_v0.fillna(False).astype(bool).sum()),
        ]
    }, index=['total', 'reviewed', 'usable', 'selected_v0'])
    display(summary)

## 5. Создание object_lexicon.csv

Список строится из уникальных `raw_name` всех собранных sim objects. Существующие русские annotations не перезаписываются при повторном запуске. Технические невидимые объекты вроде `dummy_sink_target_plane` исключаются.

In [ ]:
if inventory_df.empty:
    lexicon_df = pd.DataFrame()
    print('Object lexicon is unavailable until the inventory has been collected.')
else:
    lexicon_df = create_or_update_lexicon(DATA_DIR, inventory_df)
    print('Unique lexicon objects:', len(lexicon_df))
    display(lexicon_df.head(20))

## 6. Интерактивный review лексикона

`usable_v0 = no` ставьте, если объект трудно опознать на рендере или нельзя естественно и однозначно назвать по-русски.

In [ ]:
lexicon_reviewer = None
if lexicon_df.empty:
    print('Lexicon review is unavailable until object_lexicon.csv has been created.')
else:
    lexicon_reviewer = LexiconReviewer(lexicon_df, DATA_DIR / 'object_lexicon.csv')
    lexicon_reviewer.show()

## 7. Финальный экспорт и sanity checks

In [ ]:
if scene_reviewer is None:
    print('Nothing to export yet: inventory is empty.')
else:
    inventory_df = scene_reviewer.df
    export_inventory_dataframe(inventory_df, DATA_DIR / 'task_inventory.jsonl')
    duplicate_uids = inventory_df.task_uid[inventory_df.task_uid.duplicated()].tolist()
    missing_agent_images = [
        row.task_uid for row in inventory_df.itertuples()
        if not (DATA_DIR / row.images['agentview_rgb']).exists()
    ]
    missing_wrist_images = [
        row.task_uid for row in inventory_df.itertuples()
        if row.source['environment'] == 'LIBERO'
        and not (DATA_DIR / row.images['wrist_rgb']).exists()
    ]
    print('Rows:', len(inventory_df))
    print('Duplicate task_uid:', duplicate_uids)
    print('Missing agent images:', missing_agent_images)
    print('Missing required LIBERO wrist images:', missing_wrist_images)
    print('Selected for v0:', int(inventory_df.selected_for_v0.fillna(False).astype(bool).sum()))
    print('Saved:', DATA_DIR / 'task_inventory.jsonl')
    print('Saved:', DATA_DIR / 'object_lexicon.csv')